In [1]:
import asyncio
import httpx
import logging
import json
import sys
import os
from pathlib import Path
import time

In [2]:
import requests

In [71]:
from autogen_core.code_executor import CodeBlock,CodeExecutor
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from autogen_ext.code_executors.jupyter import JupyterCodeExecutor
import tempfile
from pathlib import Path
import venv
import asyncio
from autogen_core import CancellationToken
async def LocalCodeExecutor(codeblock_list,env=None,filedir='/oper/ch/autogen'):
    work_dir = Path(filedir)
    work_dir.mkdir(exist_ok=True)
    if not env:
        venv_dir = work_dir / ".venv"
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_builder.create(venv_dir)
        venv_context = venv_builder.ensure_directories(venv_dir)
    else:
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_context = venv_builder.ensure_directories(env)
    local_executor = LocalCommandLineCodeExecutor(work_dir=work_dir, virtual_env_context=venv_context)
    try:
        result = await local_executor.execute_code_blocks(
            code_blocks=codeblock_list,
            cancellation_token=CancellationToken(),)
        return result.output
    except Exception as e:
        return f"错误问题: {e}"

In [3]:
#sys.path.append('/oper/work/endian/intelligent_agent')

In [112]:
user_api_key={"api-key": "f16ab7cb-c2d9-4f2a-b2a9-968a0df75385"}#hao

In [116]:
user_api_key={"api-key": "5f8a621e-7f5c-4d57-a910-dabf7934c3b2"}#endian

In [131]:
response = requests.post(
    "http://localhost:8005/tabs",
    headers=user_api_key,
    json={"provider": "claude"}
)
# 获取tab_id用于后续操作
tab_id = response.json()["tab_id"]
print(tab_id)

9eb9b445-737f-4af7-8c8c-418f09adf3e7


In [130]:
response = requests.post(
    "http://localhost:8005/tabs",
    headers=user_api_key,
    json={"provider": "qwen"}
)
# 获取tab_id用于后续操作
tab_id = response.json()["tab_id"]
print(tab_id)

24e857d3-bd7c-416e-b95e-d6781fcea97e


In [132]:
# 发送消息给Claude
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": """这段代码是用来分析claude的页面内容，并生成页面元素selectors和extract_page_content。但是这个写得通用性不强，我希望不同的llm的网站的网页都能使用这个代码来自动获得页面内容和元素。请修改代码，有问题的要仔细思考并改正
""",
        "file_paths":["/oper/work/endian/intelligent_agent/page_analyzer/llm_page_analyzer.py"],
        "new_chat": True
    }
)
#print(response.json())

In [157]:
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": "~",
        "file_paths":None,
        "new_chat": False
    }
)

In [191]:
response.json()['messages'][-3]['content']['codeBlocks'][-1].keys()

dict_keys(['buttonLabel', 'code', 'language', 'version'])

In [88]:
codeblock_list=[CodeBlock(language=line['language'],code=line['code']) for line in response.json()['messages'][-3]['content']['codeBlocks'] if line['language'] in ['python','bash','sh'] ]

In [93]:
code_exe_result=await LocalCodeExecutor(codeblock_list,env='/oper/work/endian/LLM-Assistant/py310/',filedir='/oper/work/endian/intelligent_agent/page_analyzer')

In [94]:
code_exe_result

"Added to path: /oper/work/endian/intelligent_agent\nSuccessfully imported BrowserSession\n正在查找包含 chatgpt.com 的标签页...\n找到包含 chatgpt.com 的标签页: FB8A15F3DA00BE968FBA2E302E72714A\n当前所有标签页: ['FB8A15F3DA00BE968FBA2E302E72714A', '6B8FB9E909F569A8CAB0476D21BBD064']\n目标标签页句柄: FB8A15F3DA00BE968FBA2E302E72714A\n"

In [37]:
#获取所有标签页：
response = requests.get(
    "http://localhost:8005/tabs",
    headers=user_api_key
)
tabs = response.json()
print(tabs)

[{'tab_id': '46e2a78c-b426-4cac-b736-66cf713beeae', 'provider': 'claude', 'title': 'Quantum CUBO Code Example - Claude', 'url': 'https://claude.ai/chat/02512880-b168-425b-a9d1-0b95b44f092b'}]


In [26]:
import uuid

def generate_api_key():
    """生成一个随机的API密钥"""
    return str(uuid.uuid4())

# 示例使用
new_user_key = generate_api_key()

In [30]:
generate_api_key()

'f16ab7cb-c2d9-4f2a-b2a9-968a0df75385'